# Masked Pokemon Team Transformer -- Per-Slot Ensemble

Trains **five** transformers, one for each possible number of visible Pokemon (1..5). Each model
predicts the *other* 6 - K_visible slots of the team in parallel. Saved as
`masked_team_transformer_{K_visible}.pt`:
- `_1` is given 1 Pokemon, predicts the other 5
- `_2` is given 2, predicts 4
- ...
- `_5` is given 5, predicts the final 1 (equivalent to the single-slot model)

**Per-slot decoding:** each masked slot returns 5 candidate predictions. **Loss:** if the model gets
the species right (truth in top-5) on enough slots to clear the per-model threshold, the sample
contributes zero loss. Threshold = `min(3, K_remaining)`, so:
- K_remaining >= 3: max reward when >= 3 slots are species-correct
- K_remaining == 2: max reward only when both slots are correct
- K_remaining == 1: max reward only when the one slot is correct


In [1]:
import pickle
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os

## Colab Cells

Mount Google Drive and define the shared-drive directory prefixes. When running locally, comment out
the drive mount and point these prefixes at local paths instead.

In [2]:
from google.colab import drive
from google.colab import runtime
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ---- Shared-drive directory roots ----
TEAM_DIR    = '/content/drive/Shared drives/ML2 Final Project/Data/Scraped Pokemon Teams/'
VECTOR_DIR  = '/content/drive/Shared drives/ML2 Final Project/Data/Transformer Ready Vectors/'
DICT_DIR    = '/content/drive/Shared drives/ML2 Final Project/Data/Game Info Dictionaries/'
MODEL_DIR   = '/content/drive/Shared drives/ML2 Final Project/Models'
RESULTS_DIR = '/content/drive/Shared drives/ML2 Final Project/Gridsearch Results'

## Configuration

Edit this cell to change embedding sizes, transformer size, masking, and training settings.

In [4]:
# ---- Learned-embedding size presets (species / ability / item / move) ----
EMBED_CONFIGS = {
    "small":  {"species": 16, "ability": 8,  "item": 8,  "move": 16},
    "medium": {"species": 32, "ability": 16, "item": 16, "move": 32},
    "large":  {"species": 64, "ability": 32, "item": 32, "move": 64},
}
EMBED_CONFIG_NAME = "medium"          # one of: small | medium | large
EMB = EMBED_CONFIGS[EMBED_CONFIG_NAME]

# ---- Raw-data-vector dimensionalities (fixed by the prepared .pkl files) ----
SPECIES_RDV_DIM = 42
ITEM_RDV_DIM    = 6
MOVE_RDV_DIM    = 44

# ---- Moveset aggregation ----
USE_MOVE_ATTENTION = True           # if True, run MHA over the 4 moves before averaging
MOVE_ATTENTION_HEADS = 2

# ---- Transformer ----
D_MODEL          = 128
N_HEADS          = 8
N_LAYERS         = 4
DIM_FEEDFORWARD  = 256
DROPOUT          = 0.1

# ---- Masking ----
TRAIN_MASK_STRATEGY = "random"       # "random" = mask any of the 6 (augmentation); "last" = always 6th
EVAL_MASK_STRATEGY  = "last"         # held-out evaluation always predicts the 6th slot

# ---- Training ----
BATCH_SIZE   = 32
EPOCHS       = 50
LR           = 1e-3
SEED         = 42
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Debug-run override (Section 8) ----
DEBUG_RUN          = False          # if True, use a tiny 50-team sample to smoke-test the pipeline
DEBUG_TOTAL_TEAMS  = 50
DEBUG_TEST_TEAMS   = 10

# ---- Multi-Pokemon recommendation: how many candidate predictions per slot ----
# 1 = original behavior (top-1 + standard CE). n>1 = best-of-n training loss
# (zero loss when truth is among the n predictions; standard CE otherwise) and
# any-of-n accuracy at eval/inspection time.
TRAIN_PREDICTIONS = 1
TEST_PREDICTIONS  = 3

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)
print("Config:", EMBED_CONFIG_NAME, EMB, "| device:", DEVICE)

Config: medium {'species': 32, 'ability': 16, 'item': 16, 'move': 32} | device: cuda


In [5]:
# ---- Legality enforcement & sampling (Section 6 features) ----
# The model can verify a predicted Pokemon actually has the predicted ability/moves
# (using pokemon_dict) and self-correct to the best *legal* option.
ENFORCE_LEGALITY = True      # mask ability/move predictions to the predicted species' legal set

# Decoding for the SPECIES head:
#   TOP_K = 0            -> top-k disabled
#   TOP_K >= 1           -> keep K highest-logit species, softmax, sample
#   TOP_P = 0  (or >=1)  -> top-p (nucleus) disabled
#   0 < TOP_P < 1        -> keep smallest species set with cumulative prob >= TOP_P, softmax, sample
# If both are set, top-k is applied first, then top-p within it. When sampling is
# inactive the species is taken as argmax. Ability/moveset are then predicted from a
# distribution restricted to ONLY what the chosen species can legally have.
TOP_K = 0
TOP_P = 0
TEMP = 1.0

# When the top-k/p sampling pipeline is active:
#   "off"     -> never (argmax everywhere; legality still applies if ENFORCE_LEGALITY)
#   "train"   -> only while updating weights
#   "predict" -> only at inference / evaluation
#   "both"    -> training and inference
SAMPLING_PHASE = "both"

## 1. Data Loading

Raw-data-vector dictionaries (`pokemon_vectors.pkl`, `item_vectors.pkl`, `move_vectors.pkl`) come from
`prepare_raw_vectors.ipynb`. Team pools are the scraped `*_team_vectors*.pkl` files. Change `POOL_FILES`
to alter which sources feed the model.

In [6]:
with open(VECTOR_DIR + "pokemon_vectors.pkl", "rb") as f: SPECIES_RDV = pickle.load(f)
with open(VECTOR_DIR + "item_vectors.pkl",    "rb") as f: ITEM_RDV    = pickle.load(f)
with open(VECTOR_DIR + "move_vectors.pkl",    "rb") as f: MOVE_RDV    = pickle.load(f)
with open(DICT_DIR   + "ability_dict.pkl",    "rb") as f: ABILITY_DICT = pickle.load(f)
with open(DICT_DIR   + "pokemon_dict.pkl",    "rb") as f: POKEMON_DICT = pickle.load(f)

# Each pool file -> list of teams; each team -> 6 Pokemon;
# each Pokemon -> [species, ability, item, move1, move2, move3, move4]
POOL_FILES = {
    "vgcpastes":  "team_vectors.pkl",
    "vgenc":      "vgenc_team_vectors.pkl",
    "limitless":  "limitless_team_vectors.pkl",
}
TEAM_POOLS = {}
for name, path in POOL_FILES.items():
    with open(TEAM_DIR + path, "rb") as f:
        TEAM_POOLS[name] = pickle.load(f)
    print(f"{name:11s}: {len(TEAM_POOLS[name])} teams")

vgcpastes  : 769 teams
vgenc      : 2611 teams
limitless  : 6109 teams


In [7]:
for k in TEAM_POOLS.keys():
    for t in TEAM_POOLS[k]:
        if len(t) != 6:
            print(k)
            print(t)
            print(len(t))

## 2. Mega Stone Handling

If a Pokemon holds a Mega Stone, its species is renamed to the corresponding Mega form so it receives
the Mega's base stats / typing RDV. The held item is left unchanged. Charizard (X/Y) is handled
explicitly.

In [8]:
MEGA_STONE_TO_FORM = {
    "Abomasite": "Abomasnow-Mega", "Absolite": "Absol-Mega",
    "Aerodactylite": "Aerodactyl-Mega", "Aggronite": "Aggron-Mega",
    "Alakazite": "Alakazam-Mega", "Altarianite": "Altaria-Mega",
    "Ampharosite": "Ampharos-Mega", "Audinite": "Audino-Mega",
    "Banettite": "Banette-Mega", "Beedrillite": "Beedrill-Mega",
    "Blastoisinite": "Blastoise-Mega", "Cameruptite": "Camerupt-Mega",
    "Chandelurite": "Chandelure-Mega",
    "Charizardite X": "Charizard-Mega-X", "Charizardite Y": "Charizard-Mega-Y",
    "Chesnaughtite": "Chesnaught-Mega", "Chimechite": "Chimecho-Mega",
    "Clefablite": "Clefable-Mega", "Crabominite": "Crabominable-Mega",
    "Delphoxite": "Delphox-Mega", "Dragoninite": "Dragonite-Mega",
    "Drampanite": "Drampa-Mega", "Emboarite": "Emboar-Mega",
    "Excadrite": "Excadrill-Mega", "Feraligite": "Feraligatr-Mega",
    "Floettite": "Floette-Mega", "Froslassite": "Froslass-Mega",
    "Galladite": "Gallade-Mega", "Garchompite": "Garchomp-Mega",
    "Gardevoirite": "Gardevoir-Mega", "Gengarite": "Gengar-Mega",
    "Glalitite": "Glalie-Mega", "Glimmoranite": "Glimmora-Mega",
    "Golurkite": "Golurk-Mega", "Greninjite": "Greninja-Mega",
    "Gyaradosite": "Gyarados-Mega", "Hawluchanite": "Hawlucha-Mega",
    "Heracronite": "Heracross-Mega", "Houndoominite": "Houndoom-Mega",
    "Kangaskhanite": "Kangaskhan-Mega", "Lopunnite": "Lopunny-Mega",
    "Lucarionite": "Lucario-Mega", "Manectite": "Manectric-Mega",
    "Medichamite": "Medicham-Mega", "Meganiumite": "Meganium-Mega",
    "Meowsticite": "Meowstic-Mega", "Pidgeotite": "Pidgeot-Mega",
    "Pinsirite": "Pinsir-Mega", "Sablenite": "Sableye-Mega",
    "Scizorite": "Scizor-Mega", "Scovillainite": "Scovillain-Mega",
    "Sharpedonite": "Sharpedo-Mega", "Skarmorite": "Skarmory-Mega",
    "Slowbronite": "Slowbro-Mega", "Starminite": "Starmie-Mega",
    "Steelixite": "Steelix-Mega", "Tyranitarite": "Tyranitar-Mega",
    "Venusaurite": "Venusaur-Mega", "Victreebelite": "Victreebel-Mega",
}

import re
_PAREN_RE = re.compile(r"\(([^()]+)\)")

def resolve_nickname(species):
    """If `species` is not in POKEMON_DICT but contains a parenthesized
    sub-string that IS a real Pokemon name, return that name. Handles
    paste entries like "Recto/Verso (Gardevoir)" or "Warding Chime (Chimecho-Mega) (M)".
    """
    if species in POKEMON_DICT:
        return species
    for match in _PAREN_RE.findall(species):
        cand = match.strip()
        if cand in POKEMON_DICT:
            return cand
    return species

def apply_mega(pokemon):
    """Return a copy of one Pokemon list with species renamed to its Mega form
    when it holds a Mega Stone (and that Mega form exists in the species RDVs)."""
    species, ability, item = pokemon[0], pokemon[1], pokemon[2]
    mega = MEGA_STONE_TO_FORM.get(item)
    if mega is not None and mega in SPECIES_RDV:
        p = list(pokemon)
        p[0] = mega
        return p
    return list(pokemon)
def apply_variant(pokemon):
    """Return a copy of one Pokemon list with species renamed to its Mega form
    when it holds a Mega Stone (and that Mega form exists in the species RDVs)."""
    species, ability, item = pokemon[0], pokemon[1], pokemon[2]
    mega = MEGA_STONE_TO_FORM.get(item)
    if mega is not None and mega in SPECIES_RDV:
        p = list(pokemon)
        p[0] = mega
        return p
    return list(pokemon)
def normalize_team(team):
    out = []
    for p in team:
        p = list(p)
        p[0] = resolve_nickname(p[0])      # un-nickname first
        out.append(apply_mega(p))          # then promote to Mega form if holding a stone
    return out

## 2b. Alternate-name Reformatting

Catches prefix-style form names that don't match POKEMON_DICT exactly. Three layered passes:
1. **Prefix-style alias map** — `Mega Charizard Y -> Charizard-Mega-Y`, `Hisuian Zoroark -> Zoroark-Hisui`,
   `Alolan Ninetales -> Ninetales-Alola`, `Galarian Slowking -> Slowking-Galar`,
   `Paldean Tauros -> Tauros-Paldea-Combat`, `Paldean Tauros Blaze Breed -> Tauros-Paldea-Blaze`,
   `Fan Rotom -> Rotom-Fan`, `Midnight Lycanroc -> Lycanroc-Midnight`, etc.
2. **Substring fallback** — only used when the string still isn't a real name *and* the nickname
   parser already failed to recover one. Looks for any canonical name or alias appearing anywhere
   in the string (longest match wins) so junk like gender tags (`"Garchomp (M)"`, `"Scizor F"`) gets
   cleaned to the bare species. Pre-mapped aliases (`Mega Venusaur` *and* `Venusaur-Mega`) both
   count as detectable names.
3. **normalize_team** is overridden to chain: nickname -> form reformat -> substring fallback -> Mega Stone rename.

In [9]:
# ---- Toggle: case-insensitive name lookups -------------------------------
# When True, name matching ignores casing throughout the rest of the pipeline
# (resolve_nickname, reformat_form, find_pokemon_substring). This recovers
# scrape artefacts like "Kommo-O" -> "Kommo-o" without losing canonical casing
# in the output.
CASE_INSENSITIVE_NAMES = True

# ---- Prefix-style alias map ----------------------------------------------
_REGION_PREFIX = {"Alola": "Alolan", "Galar": "Galarian",
                  "Hisui": "Hisuian", "Paldea": "Paldean"}

def _build_alias_map():
    """Build {alias: canonical_POKEMON_DICT_key} for prefix-style alternate names."""
    aliases = {}
    for canonical in POKEMON_DICT:
        if canonical.endswith("-Mega-X"):
            aliases[f"Mega {canonical[:-len('-Mega-X')]} X"] = canonical
        elif canonical.endswith("-Mega-Y"):
            aliases[f"Mega {canonical[:-len('-Mega-Y')]} Y"] = canonical
        elif canonical.endswith("-Mega"):
            aliases[f"Mega {canonical[:-len('-Mega')]}"] = canonical
        for suffix, prefix in _REGION_PREFIX.items():
            tag = f"-{suffix}"
            if canonical.endswith(tag):
                aliases[f"{prefix} {canonical[:-len(tag)]}"] = canonical
            else:
                m = re.match(rf"(.+)-{suffix}-(.+)", canonical)
                if m:
                    base, sub = m.group(1), m.group(2)
                    aliases[f"{prefix} {base} {sub} Breed"] = canonical
        if canonical.startswith("Rotom-"):
            aliases[f"{canonical[len('Rotom-'):]} Rotom"] = canonical
        if canonical.startswith("Lycanroc-"):
            aliases[f"{canonical[len('Lycanroc-'):]} Lycanroc"] = canonical
    if "Tauros-Paldea-Combat" in POKEMON_DICT:
        aliases["Paldean Tauros"] = "Tauros-Paldea-Combat"
    return aliases

ALIAS_TO_CANONICAL = _build_alias_map()

# Lowercase-keyed views for case-insensitive lookups (built once, reused).
_POKEMON_DICT_LOWER = {k.lower(): k for k in POKEMON_DICT}
_ALIAS_LOWER        = {k.lower(): v for k, v in ALIAS_TO_CANONICAL.items()}

# All detectable names, sorted by length so substring search prefers the most
# specific match. Two lists: case-sensitive (raw) and case-insensitive (with
# lowercased keys + canonical values pre-resolved).
_ALL_NAMES_SORTED       = sorted(set(POKEMON_DICT) | set(ALIAS_TO_CANONICAL),
                                 key=len, reverse=True)
_ALL_NAMES_LOWER_SORTED = sorted(
    [(n.lower(), ALIAS_TO_CANONICAL.get(n, n))
     for n in set(POKEMON_DICT) | set(ALIAS_TO_CANONICAL)],
    key=lambda x: len(x[0]), reverse=True)
print(f"alias map: {len(ALIAS_TO_CANONICAL)} prefix-style aliases registered "
      f"(case_insensitive={CASE_INSENSITIVE_NAMES})")

# ---- Per-stage helpers ---------------------------------------------------
def _canonicalize(name):
    """Return the canonical POKEMON_DICT key for `name` via exact match (and
    case-insensitive fallback when CASE_INSENSITIVE_NAMES). Returns None if
    no canonical match is found."""
    if name in POKEMON_DICT:
        return name
    if name in ALIAS_TO_CANONICAL:
        return ALIAS_TO_CANONICAL[name]
    if CASE_INSENSITIVE_NAMES:
        lower = name.lower()
        if lower in _POKEMON_DICT_LOWER:
            return _POKEMON_DICT_LOWER[lower]
        if lower in _ALIAS_LOWER:
            return _ALIAS_LOWER[lower]
    return None

def reformat_form(species):
    """Exact / case-insensitive alias -> canonical. Pass-through otherwise."""
    return _canonicalize(species) or species

def resolve_nickname(species):  # OVERRIDES the mega-cell version
    """If `species` is itself a canonical name (case-insensitively allowed),
    return it. Otherwise try each parenthesized sub-string."""
    canon = _canonicalize(species)
    if canon is not None:
        return canon
    for match in _PAREN_RE.findall(species):
        canon = _canonicalize(match.strip())
        if canon is not None:
            return canon
    return species

def find_pokemon_substring(species):
    """Last-resort cleanup: scan `species` for any known name as a substring
    (longest match wins). Honors CASE_INSENSITIVE_NAMES."""
    canon = _canonicalize(species)
    if canon is not None:
        return canon
    if CASE_INSENSITIVE_NAMES:
        lower = species.lower()
        for lname, canonical in _ALL_NAMES_LOWER_SORTED:
            if lname in lower:
                return canonical
    else:
        for name in _ALL_NAMES_SORTED:
            if name in species:
                return ALIAS_TO_CANONICAL.get(name, name)
    return species

# ---- Override normalize_team to chain all four passes --------------------
# Order: parenthesized nickname -> prefix-form alias -> substring fallback
# (only when no real name + no successful nickname) -> item-based Mega rename.
def normalize_team(team):
    out = []
    for p in team:
        p = list(p)
        sp = p[0]
        sp = resolve_nickname(sp)
        sp = reformat_form(sp)
        if sp not in POKEMON_DICT:
            sp = find_pokemon_substring(sp)
        p[0] = sp
        out.append(apply_mega(p))
    return out

alias map: 82 prefix-style aliases registered (case_insensitive=True)


## 3. Vocabularies

Learned-embedding vocabularies are built from every name observed across **all** pools (so train and
held-out teams share the index space) unioned with the RDV dictionary keys. Index 0 is reserved for
`<UNK>` (covers names with no RDV / unseen entries). RDV lookups fall back to a zero vector when a
name is absent, and missing fractions are reported.

In [10]:
UNK = "<UNK>"

def build_vocab(values):
    vocab = {UNK: 0}
    for v in values:
        if v not in vocab:
            vocab[v] = len(vocab)
    return vocab

_sp, _ab, _it, _mv = set(SPECIES_RDV), set(ABILITY_DICT), set(ITEM_RDV), set(MOVE_RDV)
for pool in TEAM_POOLS.values():
    for team in pool:
        for p in normalize_team(team):
            _sp.add(p[0]); _ab.add(p[1]); _it.add(p[2])
            for m in p[3:7]: _mv.add(m)

SPECIES_VOCAB = build_vocab(sorted(_sp))
ABILITY_VOCAB = build_vocab(sorted(_ab))
ITEM_VOCAB    = build_vocab(sorted(_it))
MOVE_VOCAB    = build_vocab(sorted(_mv))
N_SPECIES, N_ABILITY = len(SPECIES_VOCAB), len(ABILITY_VOCAB)
N_ITEM, N_MOVE       = len(ITEM_VOCAB), len(MOVE_VOCAB)
print(f"vocab sizes -> species {N_SPECIES}, ability {N_ABILITY}, item {N_ITEM}, move {N_MOVE}")

ZERO_SPECIES_RDV = [0.0] * SPECIES_RDV_DIM
ZERO_ITEM_RDV    = [0.0] * ITEM_RDV_DIM
ZERO_MOVE_RDV    = [0.0] * MOVE_RDV_DIM

# RDV coverage diagnostic over normalized teams
tot = miss_s = miss_i = miss_m = 0
for pool in TEAM_POOLS.values():
    for team in pool:
        for p in normalize_team(team):
            tot += 1
            if p[0] not in SPECIES_RDV: miss_s += 1
            if p[2] not in ITEM_RDV:    miss_i += 1
            for m in p[3:7]:
                if m not in MOVE_RDV:   miss_m += 1
print(f"RDV miss rate -> species {miss_s/tot:.1%}, item {miss_i/tot:.1%}, move {miss_m/(tot*4):.1%}")

vocab sizes -> species 267, ability 269, item 209, move 683
RDV miss rate -> species 5.7%, item 0.9%, move 1.2%


## 4. Team -> Tensor Conversion

`team_to_tensors` maps a raw nested-list team (post Mega-rename) to index tensors and RDV tensors by
looking each name up in its vocabulary / RDV dictionary. `MaskedTeamDataset` then picks the masked
slot per sample and exposes the prediction targets.

In [11]:
def team_to_tensors(team):
    """team: list of 6 Pokemon lists (already Mega-normalized).
    Returns a dict of tensors describing all 6 Pokemon."""
    sp_idx, ab_idx, it_idx = [], [], []
    mv_idx, sp_rdv, it_rdv, mv_rdv = [], [], [], []
    for p in team:
        species, ability, item = p[0], p[1], p[2]
        moves = list(p[3:7]) + [""] * (4 - len(p[3:7]))
        sp_idx.append(SPECIES_VOCAB.get(species, 0))
        ab_idx.append(ABILITY_VOCAB.get(ability, 0))
        it_idx.append(ITEM_VOCAB.get(item, 0))
        mv_idx.append([MOVE_VOCAB.get(m, 0) for m in moves])
        sp_rdv.append(SPECIES_RDV.get(species, ZERO_SPECIES_RDV))
        it_rdv.append(ITEM_RDV.get(item, ZERO_ITEM_RDV))
        mv_rdv.append([MOVE_RDV.get(m, ZERO_MOVE_RDV) for m in moves])
    return {
        "species_idx": torch.tensor(sp_idx, dtype=torch.long),
        "ability_idx": torch.tensor(ab_idx, dtype=torch.long),
        "item_idx":    torch.tensor(it_idx, dtype=torch.long),
        "move_idx":    torch.tensor(mv_idx, dtype=torch.long),
        "species_rdv": torch.tensor(sp_rdv, dtype=torch.float),
        "item_rdv":    torch.tensor(it_rdv, dtype=torch.float),
        "move_rdv":    torch.tensor(mv_rdv, dtype=torch.float),
    }

class MaskedTeamDataset(Dataset):
    """Masks `n_masked` slots out of the 6 team positions and returns per-slot targets.

    mask_strategy:
      "random" -> sample n_masked distinct positions uniformly each __getitem__
      "last"   -> always mask the trailing n_masked positions (deterministic eval)
    """
    def __init__(self, teams, mask_strategy, n_masked=1):
        assert 1 <= n_masked <= 6, "n_masked must be in [1, 6]"
        self.teams = [normalize_team(t) for t in teams]
        self.mask_strategy = mask_strategy
        self.n_masked = n_masked

    def __len__(self):
        return len(self.teams)

    def __getitem__(self, i):
        team = self.teams[i]
        t = team_to_tensors(team)
        K = self.n_masked
        if self.mask_strategy == "random":
            mpos = sorted(random.sample(range(6), K))
        else:                                   # "last"
            mpos = list(range(6 - K, 6))
        mpos = torch.tensor(mpos, dtype=torch.long)             # [K]
        t["mask_pos"]  = mpos
        t["y_species"] = t["species_idx"][mpos].clone()          # [K]
        t["y_ability"] = t["ability_idx"][mpos].clone()
        t["y_item"]    = t["item_idx"][mpos].clone()
        mv_multi = torch.zeros(K, N_MOVE)
        for k_i, slot in enumerate(mpos.tolist()):
            for mi in t["move_idx"][slot].tolist():
                mv_multi[k_i, mi] = 1.0
        t["y_moves"] = mv_multi                                  # [K, N_MOVE]
        return t

## 5. Train / Test Split

Edit this section to change the pool or the split. By default the **test set is a random 20% of the
`limitless` teams**; everything else (`vgcpastes`, `vgenc`, and the remaining 80% of `limitless`)
is training. The debug override (Section 8) instead carves a 50-team sample.

In [12]:
TEST_SOURCE   = "limitless"
TEST_FRACTION = 0.20

def make_split(team_pools, test_source=TEST_SOURCE, test_fraction=TEST_FRACTION, seed=SEED):
    rng = random.Random(seed)
    src = list(team_pools[test_source])
    idx = list(range(len(src)))
    rng.shuffle(idx)
    n_test = int(round(len(src) * test_fraction))
    test_ids = set(idx[:n_test])
    test_teams  = [src[i] for i in idx[:n_test]]
    train_teams = [src[i] for i in idx[n_test:]]
    for name, pool in team_pools.items():
        if name == test_source:
            continue
        train_teams.extend(pool)
    rng.shuffle(train_teams)
    return train_teams, test_teams

def make_debug_split(team_pools, total=DEBUG_TOTAL_TEAMS, n_test=DEBUG_TEST_TEAMS,
                     source=TEST_SOURCE, seed=SEED):
    rng = random.Random(seed)
    src = list(team_pools[source])
    rng.shuffle(src)
    sample = src[:total]
    return sample[n_test:], sample[:n_test]

if DEBUG_RUN:
    train_teams, test_teams = make_debug_split(TEAM_POOLS)
else:
    train_teams, test_teams = make_split(TEAM_POOLS)
print(f"train teams: {len(train_teams)} | test teams: {len(test_teams)}")

train teams: 8267 | test teams: 1222


In [13]:
full_teams = train_teams + test_teams

In [14]:
team_mon_list = [[p[0] for p in t] for t in full_teams]
names_in_teams = set([item for sublist in team_mon_list for item in sublist])

In [15]:
print("Appears in team DB but not in Pokemon Dict Keys")
for n in names_in_teams:
    if n not in POKEMON_DICT.keys():
        print(n)
print("Appears in Pokemon Dict Keys but not in team DB")
for k in POKEMON_DICT.keys():
    if k not in names_in_teams:
        print(k)

Appears in team DB but not in Pokemon Dict Keys
Mega Aerodactyl
Victreebel-Mega (M)
Torkoal (M)
Basculegion-F
Volcarona (M)
Aegislash (M)
Solace (Garchomp)
Emperor (Incineroar)
Charizard (M)
Swanging (Incineroar) (M)
Howler (Talonflame)
Mega Glimmora
Galarian Slowking
Kingambit (F)
Sylveon (F)
Hisuian Goodra
Mega Venusaur
Riverwalker (Araquanid) (M)
Mega Sharpedo
Emperor (Kingambit)
Basculegion (M)
I CONNECT (Rotom-Heat)
Garchomp-Mega (M)
Recto/Verso (Gardevoir)
Floréclat (Glimmora-Mega) (F)
Méga-Soléil (Meganium-Mega) (F)
Mega Starmie
Hisuian Samurott
Alolan Raichu
Mega Gengar
Frost Rotom
Don't hit me. (Runerigus) (M)
Empress (Lopunny-Mega)
Hatterene (F)
Floette-Mega (F)
Samurott-Hisui (M)
Corviknight (M)
Chevalier (Basculegion)
Lycanroc Dusk
Mega Lopunny
Kingambit (M)
Diva (Kingambit)
Mega Scovillain
Mega Gardevoir
Tauros-Paldea-Aqua (M)
Mega Manectric
Meowscarada (F)
Gardevoir (F)
Kangaskhan (F)
Hisuian Avalugg
Basculegion-F (F)
Alolan Ninetales
Kommo-O
Volcarona (F)
Hisuian Arcanin

In [16]:
for k in POKEMON_DICT.keys():
    if "Lycanroc" in k:
        print(k)

Lycanroc
Lycanroc-Midnight
Lycanroc-Dusk


## 6. Model

Per-Pokemon vector =
`[species_emb | species_rdv | ability_emb | item_emb | item_rdv | moveset_emb | moveset_rdv]`.
The masked slot's vector is replaced by a learned mask token. Vectors are projected to `D_MODEL`,
passed through a Transformer encoder (order-free / no positional encoding, since a team is a set),
and the masked slot's output feeds four prediction heads.

In [17]:
class MaskedTeamTransformer(nn.Module):
    """Masked team model that handles any number of masked slots in parallel.
    All tensor shapes for the masked-slot heads are [B, K, ...] where K is the
    number of masked positions per sample (constant within a dataset).
    """
    def __init__(self):
        super().__init__()
        self.species_emb = nn.Embedding(N_SPECIES, EMB["species"])
        self.ability_emb = nn.Embedding(N_ABILITY, EMB["ability"])
        self.item_emb    = nn.Embedding(N_ITEM,    EMB["item"])
        self.move_emb    = nn.Embedding(N_MOVE,    EMB["move"])

        self.use_move_attn = USE_MOVE_ATTENTION
        if self.use_move_attn:
            self.move_attn_emb = nn.MultiheadAttention(
                EMB["move"], MOVE_ATTENTION_HEADS, batch_first=True)
            self.move_attn_rdv = nn.MultiheadAttention(
                MOVE_RDV_DIM, MOVE_ATTENTION_HEADS, batch_first=True)

        self.input_dim = (EMB["species"] + SPECIES_RDV_DIM + EMB["ability"]
                          + EMB["item"] + ITEM_RDV_DIM
                          + EMB["move"] + MOVE_RDV_DIM)
        self.mask_token = nn.Parameter(torch.randn(self.input_dim) * 0.02)
        self.input_proj = nn.Linear(self.input_dim, D_MODEL)

        enc = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=N_LAYERS)

        self.head_species = nn.Linear(D_MODEL, N_SPECIES)
        self.head_ability = nn.Linear(D_MODEL, N_ABILITY)
        self.head_item    = nn.Linear(D_MODEL, N_ITEM)
        self.head_moves   = nn.Linear(D_MODEL, N_MOVE)

        # ---- Ability/move legality from POKEMON_DICT ----
        ability_legal = torch.zeros(N_SPECIES, N_ABILITY, dtype=torch.bool)
        move_legal    = torch.zeros(N_SPECIES, N_MOVE,    dtype=torch.bool)
        for name, sidx in SPECIES_VOCAB.items():
            entry = POKEMON_DICT.get(name)
            if entry is None:
                ability_legal[sidx] = True
                move_legal[sidx]    = True
                continue
            abils = entry.get("ability")
            abils = [abils] if isinstance(abils, str) else list(abils)
            a_idx = [ABILITY_VOCAB[a] for a in abils if a in ABILITY_VOCAB]
            if a_idx:
                ability_legal[sidx, a_idx] = True
            else:
                ability_legal[sidx] = True
            m_idx = [MOVE_VOCAB[m] for m in entry.get("moves", []) if m in MOVE_VOCAB]
            if m_idx:
                move_legal[sidx, m_idx] = True
            else:
                move_legal[sidx] = True
        self.register_buffer("ability_legal", ability_legal)
        self.register_buffer("move_legal",    move_legal)

        # ---- Team-construction constraint tables (same as single-slot model) ----
        mega_forms    = set(MEGA_STONE_TO_FORM.values())
        form_to_stone = {v: k for k, v in MEGA_STONE_TO_FORM.items()}
        is_mega   = torch.zeros(N_SPECIES, dtype=torch.bool)
        fts_item  = torch.full((N_SPECIES,), -1, dtype=torch.long)
        family_id = torch.zeros(N_SPECIES, dtype=torch.long)
        fam_lookup = {}
        for name, sidx in SPECIES_VOCAB.items():
            base = name.rsplit("-Mega", 1)[0] if name in mega_forms else name
            family_id[sidx] = fam_lookup.setdefault(base, len(fam_lookup))
            if name in mega_forms:
                is_mega[sidx] = True
                stone = form_to_stone.get(name)
                if stone in ITEM_VOCAB:
                    fts_item[sidx] = ITEM_VOCAB[stone]
        self.register_buffer("is_mega", is_mega)
        self.register_buffer("family_id", family_id)
        self.register_buffer("form_to_stone_item", fts_item)

        # ---- Mega Stone Assertion: each Mega Stone is legal only for its
        # base + Mega form. Non-Mega-Stone items remain legal for all species.
        item_legal = torch.ones(N_SPECIES, N_ITEM, dtype=torch.bool)
        for stone, form in MEGA_STONE_TO_FORM.items():
            if stone not in ITEM_VOCAB:
                continue
            iidx = ITEM_VOCAB[stone]
            base_name = form.rsplit("-Mega", 1)[0]
            legal_sp = set()
            if form in SPECIES_VOCAB:      legal_sp.add(SPECIES_VOCAB[form])
            if base_name in SPECIES_VOCAB: legal_sp.add(SPECIES_VOCAB[base_name])
            disallow = torch.ones(N_SPECIES, dtype=torch.bool)
            for s in legal_sp: disallow[s] = False
            item_legal[disallow, iidx] = False
        self.register_buffer("item_legal", item_legal)

    # ---------- shared encoding ----------
    def _moveset(self, move_idx, move_rdv):
        B = move_idx.size(0)
        me = self.move_emb(move_idx)
        if self.use_move_attn:
            e = me.reshape(B * 6, 4, EMB["move"])
            e, _ = self.move_attn_emb(e, e, e)
            me = e.reshape(B, 6, 4, EMB["move"])
            r = move_rdv.reshape(B * 6, 4, MOVE_RDV_DIM)
            r, _ = self.move_attn_rdv(r, r, r)
            move_rdv = r.reshape(B, 6, 4, MOVE_RDV_DIM)
        return me.mean(dim=2), move_rdv.mean(dim=2)

    def forward(self, batch):
        sp = self.species_emb(batch["species_idx"])
        ab = self.ability_emb(batch["ability_idx"])
        it = self.item_emb(batch["item_idx"])
        ms_emb, ms_rdv = self._moveset(batch["move_idx"], batch["move_rdv"])
        x = torch.cat([sp, batch["species_rdv"], ab,
                       it, batch["item_rdv"], ms_emb, ms_rdv], dim=-1)
        # x: [B, 6, input_dim]

        B = x.size(0)
        mpos = batch["mask_pos"]                                    # [B, K]
        K = mpos.size(1)
        b_idx = torch.arange(B, device=x.device).unsqueeze(-1).expand(-1, K)  # [B, K]
        x = x.clone()
        x[b_idx, mpos] = self.mask_token                            # broadcast mask
        h = self.encoder(self.input_proj(x))                        # [B, 6, D]
        hm = h[b_idx, mpos]                                         # [B, K, D]
        return {
            "species": self.head_species(hm),                       # [B, K, N_SP]
            "ability": self.head_ability(hm),                       # [B, K, N_AB]
            "item":    self.head_item(hm),                          # [B, K, N_IT]
            "moves":   self.head_moves(hm),                         # [B, K, N_MV]
        }

    # ---------- decoding ----------
    @staticmethod
    def _filter_logits(logits, top_k, top_p):
        logits = logits.clone()
        if top_k and top_k >= 1:
            k = min(int(top_k), logits.size(-1))
            kth = logits.topk(k, dim=-1).values[..., -1, None]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        if top_p and 0.0 < top_p < 1.0:
            s_logits, s_idx = torch.sort(logits, descending=True, dim=-1)
            cum = s_logits.softmax(-1).cumsum(-1)
            s_remove = cum > top_p
            s_remove[..., 1:] = s_remove[..., :-1].clone()
            s_remove[..., 0] = False
            remove = torch.zeros_like(s_remove).scatter(-1, s_idx, s_remove)
            logits = logits.masked_fill(remove, float("-inf"))
        return logits

    @staticmethod
    def _sampling_active(phase):
        if not ((TOP_K and TOP_K >= 1) or (TOP_P and 0.0 < TOP_P < 1.0)):
            return False
        return SAMPLING_PHASE == "both" or SAMPLING_PHASE == phase

    @torch.no_grad()
    def predict(self, batch, phase="predict", n_predictions=None, n_moves=4):
        """Per-slot top-N candidates with team-construction + Mega-stone assertions.

        Returns dict with:
          species [B, K, n]   ability [B, K, n]   item [B, K, n]
          moves   list[B] of list[K] of list[n] of LongTensor[k_moves]
          logits  raw model output for that batch
        Constraints applied per slot (independent across slots): no duplicate /
        base<->Mega of any VISIBLE Pokemon, no Mega when >=2 visible are Megas,
        ability/move legality if ENFORCE_LEGALITY, Mega-Stone legality always,
        Mega form forces its stone as item.
        """
        if n_predictions is None:
            n_predictions = TEST_PREDICTIONS
        out = self.forward(batch)
        B, K, S = out["species"].shape

        # ----- visible slots / species per sample -----
        mpos = batch["mask_pos"]                                            # [B, K]
        all_pos = torch.arange(6, device=mpos.device).unsqueeze(0).expand(B, -1)
        is_masked = (all_pos.unsqueeze(-1) == mpos.unsqueeze(1)).any(-1)    # [B, 6]
        is_visible = ~is_masked
        n_vis = 6 - K
        vis_sp = batch["species_idx"][is_visible].view(B, n_vis)            # [B, 6-K]

        # ----- species constraints (shared across the K slots) -----
        vis_fam = self.family_id[vis_sp]                                    # [B, 6-K]
        forbid = (self.family_id.view(1, S, 1) == vis_fam.view(B, 1, n_vis)).any(-1)
        two_megas = self.is_mega[vis_sp].sum(-1) >= 2                       # [B]
        forbid = forbid | (two_megas.view(B, 1) & self.is_mega.view(1, S))  # [B, S]

        sp_logits = out["species"].masked_fill(forbid.unsqueeze(1), float("-inf"))
        dead = torch.isinf(sp_logits).all(-1)                               # [B, K]
        if dead.any():
            sp_logits[dead] = out["species"][dead]

        active = self._sampling_active(phase) and n_predictions == 1
        n_eff = min(n_predictions, S)
        if active:
            probs = F.softmax(self._filter_logits(sp_logits, TOP_K, TOP_P) / TEMP, -1)
            sp_cands = torch.multinomial(probs.reshape(B * K, S), 1).reshape(B, K, 1)
        else:
            sp_cands = sp_logits.topk(n_eff, dim=-1).indices                # [B, K, n]

        n = sp_cands.size(-1)
        ab_cands = torch.zeros_like(sp_cands)
        it_cands = torch.zeros_like(sp_cands)
        mv_cands = [[[None] * n for _ in range(K)] for _ in range(B)]

        for c in range(n):
            sp_c = sp_cands[..., c]                                         # [B, K]
            ab_logits = out["ability"].clone()
            mv_logits = out["moves"].clone()
            it_logits = out["item"].clone()
            if ENFORCE_LEGALITY:
                ab_logits = ab_logits.masked_fill(~self.ability_legal[sp_c], float("-inf"))
                mv_logits = mv_logits.masked_fill(~self.move_legal[sp_c],    float("-inf"))
            it_logits = it_logits.masked_fill(~self.item_legal[sp_c], float("-inf"))

            ab_cands[..., c] = ab_logits.argmax(-1)
            it_pred = it_logits.argmax(-1)
            forced = self.form_to_stone_item[sp_c]
            it_cands[..., c] = torch.where(forced >= 0, forced, it_pred)

            for b in range(B):
                for k in range(K):
                    legal = torch.isfinite(mv_logits[b, k])
                    kk = int(min(n_moves, int(legal.sum().item()))) or 1
                    mv_cands[b][k][c] = mv_logits[b, k].topk(kk).indices

        return {"species": sp_cands, "ability": ab_cands, "item": it_cands,
                "moves": mv_cands, "logits": out}

    # ----- Hidden-state / embedding extraction -----
    @torch.no_grad()
    def get_embedding(self, name, kind):
        table = {"species": (SPECIES_VOCAB, self.species_emb),
                 "ability": (ABILITY_VOCAB, self.ability_emb),
                 "item":    (ITEM_VOCAB,    self.item_emb),
                 "move":    (MOVE_VOCAB,    self.move_emb)}
        if kind not in table:
            raise ValueError(f"kind must be one of {list(table)}, got {kind!r}")
        vocab, emb = table[kind]
        if name not in vocab:
            raise KeyError(f"{name!r} not found in {kind} vocab")
        idx = torch.tensor(vocab[name], device=emb.weight.device)
        return emb(idx).detach().clone()

## 7. Training & Validation Pipeline

In [18]:
ce  = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

def slot_threshold(K):
    """Per the spec: max reward when >=3 slots are species-correct for K>=3,
    or when ALL slots are correct for K in {1, 2}."""
    return min(3, K)

def compute_loss(model, out, batch, phase):
    """Multi-slot loss with sample-level reward gating.

    Per (sample, slot): CE on species/ability/item + mean BCE on moves.
    Sum across the K slots -> per-sample loss. If the model gets the species
    right (truth among top-`n_predictions`) on >= threshold(K) slots, the whole
    sample's loss is zeroed (max reward). Otherwise the standard summed loss
    contributes a gradient."""
    n = TRAIN_PREDICTIONS if phase == "train" else TEST_PREDICTIONS
    B, K = batch["y_species"].shape
    T = slot_threshold(K)
    S = out["species"].size(-1)

    n_eff = min(n, S)
    sp_topn = out["species"].topk(n_eff, dim=-1).indices                    # [B, K, n]
    sp_correct_slot = (sp_topn == batch["y_species"].unsqueeze(-1)).any(-1) # [B, K]
    sp_correct_count = sp_correct_slot.sum(-1)                              # [B]
    in_max_reward = sp_correct_count >= T                                   # [B]

    sp_ce = F.cross_entropy(out["species"].reshape(B * K, -1),
                            batch["y_species"].reshape(B * K),
                            reduction="none").reshape(B, K)
    ab_ce = F.cross_entropy(out["ability"].reshape(B * K, -1),
                            batch["y_ability"].reshape(B * K),
                            reduction="none").reshape(B, K)
    it_ce = F.cross_entropy(out["item"].reshape(B * K, -1),
                            batch["y_item"].reshape(B * K),
                            reduction="none").reshape(B, K)
    mv_bce = F.binary_cross_entropy_with_logits(
        out["moves"], batch["y_moves"], reduction="none").mean(-1)          # [B, K]

    per_slot = sp_ce + ab_ce + it_ce + mv_bce                               # [B, K]
    per_sample = per_slot.sum(-1)                                           # [B]
    keep = (~in_max_reward).float()
    return (per_sample * keep).mean()

def to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

@torch.no_grad()
def evaluate(model, loader, device, phase="predict"):
    """Per-slot accuracy + max-reward sample rate. Accuracy is any-of-N at the
    slot level (truth in top-`TEST_PREDICTIONS` candidates for that slot)."""
    n = TEST_PREDICTIONS
    model.eval()
    cnt_samples = cnt_slots = 0
    correct = {"species": 0, "ability": 0, "item": 0}
    move_recall = 0.0
    max_reward = 0
    total_loss = 0.0
    for batch in loader:
        batch = to_device(batch, device)
        out = model.forward(batch)
        B, K = batch["y_species"].shape
        total_loss += compute_loss(model, out, batch, phase).item() * B
        pred = model.predict(batch, phase=phase, n_predictions=n)
        for key in correct:
            hit = (pred[key] == batch[f"y_{key}"].unsqueeze(-1)).any(-1)
            correct[key] += hit.sum().item()
        for b in range(B):
            for k in range(K):
                true_idx = set(batch["y_moves"][b, k].nonzero(as_tuple=True)[0].tolist())
                best = 0.0
                for mv in pred["moves"][b][k]:
                    overlap = len(set(mv.tolist()) & true_idx)
                    best = max(best, overlap / max(len(true_idx), 1))
                move_recall += best
        sp_hit = (pred["species"] == batch["y_species"].unsqueeze(-1)).any(-1)  # [B,K]
        T = slot_threshold(K)
        max_reward += (sp_hit.sum(-1) >= T).sum().item()
        cnt_samples += B
        cnt_slots   += B * K
    return {
        "loss":            total_loss / max(cnt_samples, 1),
        "species_acc":     correct["species"] / max(cnt_slots, 1),
        "ability_acc":     correct["ability"] / max(cnt_slots, 1),
        "item_acc":        correct["item"]    / max(cnt_slots, 1),
        "move_recall":     move_recall / max(cnt_slots, 1),
        "max_reward_rate": max_reward / max(cnt_samples, 1),
    }

def train(model, train_ds, test_ds, epochs=EPOCHS, batch_size=BATCH_SIZE,
          lr=LR, device=DEVICE, label=""):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    vl = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    for ep in range(1, epochs + 1):
        model.train()
        run = 0.0
        for batch in tl:
            batch = to_device(batch, device)
            opt.zero_grad()
            loss = compute_loss(model, model.forward(batch), batch, "train")
            loss.backward()
            opt.step()
            run += loss.item() * batch["y_species"].size(0)
        val = evaluate(model, vl, device, phase="predict")
        print(f"{label}epoch {ep:2d} | train_loss {run/len(train_ds):.4f} "
              f"| val_loss {val['loss']:.4f} | mr {val['max_reward_rate']:.2f} "
              f"| spc {val['species_acc']:.2f} ab {val['ability_acc']:.2f} "
              f"it {val['item_acc']:.2f} mv {val['move_recall']:.2f}")
    return model

## 7.5 Gridsearch — not supported in this notebook

The per-slot ensemble trains 5 models per run, which doesn't compose cleanly with the architecture /
behavior gridsearches from the single-slot notebook. Use the single-slot notebook for gridsearches.

## 8. Per-Slot Training Loop

Trains 5 transformers in sequence -- one for each value of `K_visible` in 1..5. Each model is
instantiated fresh, trained on a dataset that masks `6 - K_visible` slots per team, and saved to
Drive as `masked_team_transformer_{K_visible}.pt`. Set `K_VISIBLE_RANGE` if you only want to train a
subset (e.g. `[5]` to retrain just the original single-slot model).

In [ ]:
K_VISIBLE_RANGE = [1, 2, 3, 4, 5]   # which masking ratios to train; [1..5]
PREDICTIONS_PER_SLOT = 5             # candidate predictions per slot (spec: "predict 5 pokemon")

TRAIN_PREDICTIONS = PREDICTIONS_PER_SLOT
TEST_PREDICTIONS  = PREDICTIONS_PER_SLOT

trained_models = {}
for K_VIS in K_VISIBLE_RANGE:
    n_masked = 6 - K_VIS
    T = slot_threshold(n_masked)
    set_seed(SEED)
    train_ds = MaskedTeamDataset(train_teams, TRAIN_MASK_STRATEGY, n_masked=n_masked)
    test_ds  = MaskedTeamDataset(test_teams,  EVAL_MASK_STRATEGY,  n_masked=n_masked)
    print(f"\n{'='*70}")
    print(f"TRAINING K_visible={K_VIS}  (n_masked={n_masked}, max-reward threshold={T})")
    print(f"  datasets: train {len(train_ds)} | test {len(test_ds)}")
    print(f"{'='*70}")

    model = MaskedTeamTransformer()
    if K_VIS == K_VISIBLE_RANGE[0]:
        print("per-Pokemon input dim:", model.input_dim)
        _sample = next(iter(DataLoader(train_ds, batch_size=4)))
        _out = model(to_device(model.cpu() and _sample, "cpu"))
        print("output shapes:", {k: tuple(v.shape) for k, v in _out.items()})

    model = train(model, train_ds, test_ds, epochs=EPOCHS, label=f"[K={K_VIS}] ")
    print(f"[K={K_VIS}] Final:",
          evaluate(model, DataLoader(test_ds, batch_size=BATCH_SIZE), DEVICE))
    trained_models[K_VIS] = (model, test_ds)

## 9. Saving / Loading the Model

`save_checkpoint` bundles the trained weights together with the architecture config and the
vocabularies, so the model can be rebuilt for inference later without re-deriving anything.
`load_checkpoint` reconstructs it. The debug-trained model is saved below.

In [23]:
def k_model_path(k_vis):
    return os.path.join(MODEL_DIR, f"masked_team_transformer_{k_vis}.pt")

def save_checkpoint(model, path):
    torch.save({
        "state_dict": model.state_dict(),
        "config": {
            "EMB": EMB,
            "SPECIES_RDV_DIM": SPECIES_RDV_DIM,
            "ITEM_RDV_DIM": ITEM_RDV_DIM,
            "MOVE_RDV_DIM": MOVE_RDV_DIM,
            "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS,
            "DIM_FEEDFORWARD": DIM_FEEDFORWARD, "DROPOUT": DROPOUT,
            "USE_MOVE_ATTENTION": USE_MOVE_ATTENTION,
            "MOVE_ATTENTION_HEADS": MOVE_ATTENTION_HEADS,
            "embed_config_name": EMBED_CONFIG_NAME,
        },
        "vocabs": {"species": SPECIES_VOCAB, "ability": ABILITY_VOCAB,
                   "item": ITEM_VOCAB, "move": MOVE_VOCAB},
    }, path)
    print(f"saved -> {path} ({os.path.getsize(path)/1e6:.2f} MB)")

def load_checkpoint(path, device=DEVICE):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = MaskedTeamTransformer()
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    print(f"loaded <- {path} (config: {ckpt['config']['embed_config_name']})")
    return model, ckpt

# Save every trained model under its K_visible suffix
for K_VIS, (m, _) in trained_models.items():
    save_checkpoint(m, k_model_path(K_VIS))

saved checkpoint -> /content/drive/Shared drives/ML2 Final Project/Models/masked_team_transformer.pt (3.53 MB)
loaded checkpoint <- /content/drive/Shared drives/ML2 Final Project/Models/masked_team_transformer.pt (trained config: medium)
reloaded model eval: {'loss': 15.41190758313368, 'species_acc': 0.5875613747954174, 'ability_acc': 0.532733224222586, 'item_acc': 0.4705400981996727, 'move_recall': 0.57569558101473}


## 10. Inspect Sample Predictions

Set `N_SAMPLES` to how many random validation teams you want to inspect. For each, the masked slot's
predicted species / ability / item / moveset is shown next to the ground truth. Moves are multi-label,
so the top-k predictions (k = number of true moves) are shown.

In [24]:
N_SAMPLES        = 3   # validation teams to inspect per model
INSPECT_K_RANGE  = K_VISIBLE_RANGE

INV_SPECIES = {v: k for k, v in SPECIES_VOCAB.items()}
INV_ABILITY = {v: k for k, v in ABILITY_VOCAB.items()}
INV_ITEM    = {v: k for k, v in ITEM_VOCAB.items()}
INV_MOVE    = {v: k for k, v in MOVE_VOCAB.items()}

@torch.no_grad()
def show_predictions(model, dataset, K_vis, n=N_SAMPLES,
                     n_predictions=None, device=DEVICE):
    if n_predictions is None:
        n_predictions = TEST_PREDICTIONS
    model.eval()
    n = min(n, len(dataset))
    idxs = random.sample(range(len(dataset)), n)
    for s_i, di in enumerate(idxs, 1):
        sample = dataset[di]
        batch = {k: v.unsqueeze(0).to(device) for k, v in sample.items()}
        pred = model.predict(batch, phase="predict", n_predictions=n_predictions)
        mpos = sample["mask_pos"].tolist()
        team = dataset.teams[di]
        visible = [team[j][0] for j in range(6) if j not in mpos]
        K = len(mpos)
        T = slot_threshold(K)

        sp_correct_slots = 0
        per_slot_lines = []
        for k in range(K):
            slot = mpos[k]
            sp_cs = [INV_SPECIES[i] for i in pred["species"][0, k].tolist()]
            ab_cs = [INV_ABILITY[i] for i in pred["ability"][0, k].tolist()]
            it_cs = [INV_ITEM[i]    for i in pred["item"][0, k].tolist()]
            mv_cs = [[INV_MOVE[i] for i in m.tolist()] for m in pred["moves"][0][k]]
            t_sp = INV_SPECIES[sample["y_species"][k].item()]
            t_ab = INV_ABILITY[sample["y_ability"][k].item()]
            t_it = INV_ITEM[sample["y_item"][k].item()]
            t_mv = [INV_MOVE[i] for i in
                    sample["y_moves"][k].nonzero(as_tuple=True)[0].tolist()]
            sp_hit = t_sp in sp_cs
            if sp_hit: sp_correct_slots += 1
            mark = lambda ok: "OK " if ok else "X  "
            per_slot_lines.append(f"  slot {slot}:")
            per_slot_lines.append(f"    species : {mark(sp_hit)}preds={sp_cs}   true={t_sp}")
            per_slot_lines.append(f"    ability : {mark(t_ab in ab_cs)}preds={ab_cs}   true={t_ab}")
            per_slot_lines.append(f"    item    : {mark(t_it in it_cs)}preds={it_cs}   true={t_it}")
            per_slot_lines.append(f"    moves   : preds[0]={mv_cs[0]}")
            per_slot_lines.append(f"              true   ={t_mv}")
        reached = sp_correct_slots >= T
        print(f"=== Sample {s_i}  (team #{di}, K_vis={K_vis}, "
              f"max-reward: {sp_correct_slots}/{K} correct  threshold={T}  "
              f"-> {'REACHED' if reached else 'missed'}) ===")
        print("  given     :", ", ".join(visible) if visible else "(none)")
        for line in per_slot_lines:
            print(line)
        print()

if "trained_models" in globals() and trained_models:
    for K_VIS in INSPECT_K_RANGE:
        if K_VIS not in trained_models:
            continue
        m, ds_k = trained_models[K_VIS]
        print(f"\n>>>>>>>>>> Inspecting model K_visible={K_VIS} <<<<<<<<<<")
        show_predictions(m, ds_k, K_VIS, N_SAMPLES)
else:
    print("skip: no trained models in memory.")

=== Sample 1  (test team #363, masked slot 5, n_preds=3) ===
  given 5  : Bellibolt, Delphox-Mega, Garchomp, Sinistcha, Incineroar
  species  : X  preds=['Milotic', 'Gyarados', 'Primarina']   true=Froslass-Mega
  ability  : X  preds=['Competitive', 'Intimidate', 'Liquid Voice']   true=Snow Cloak
  item     : X  preds=['Leftovers', 'Leftovers', 'Leftovers']   true=Froslassite
  moves #1: pred=['Protect', 'Recover', 'Scald', 'Hypnosis']
  moves #2: pred=['Protect', 'Scald', 'Icy Wind', 'Waterfall']
  moves #3: pred=['Moonblast', 'Protect', 'Calm Mind', 'Dazzling Gleam']
             true=['Aurora Veil', 'Blizzard', 'Protect', 'Will-O-Wisp']

=== Sample 2  (test team #1138, masked slot 5, n_preds=3) ===
  given 5  : Zoroark-Hisui, Lucario-Mega, Milotic, Farigiraf, Aerodactyl
  species  : X  preds=['Sneasler', 'Garchomp', 'Meganium-Mega']   true=Tyranitar
  ability  : X  preds=['Unburden', 'Rough Skin', 'Mega Sol']   true=Sand Stream
  item     : X  preds=['White Herb', 'White Herb', 'Mega